## Data Loading 

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("IMDB Dataset.csv")

In [3]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [5]:
df.drop_duplicates(inplace=True)

In [6]:
df.shape

(49582, 2)

## Text Pre-processing

###  1. Converting to lowercase

In [7]:
df["review"] = df["review"].str.lower()

### 2. Removing URL's

In [8]:
import re

def remove_urls(text):
    text = re.sub(r"http\S+", "", text)
    return text

df["review"]=df["review"].apply(remove_urls)

### 3. Removing Punctuations

In [9]:
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]", "", text)
    return text

df["review"]=df["review"].apply(remove_punctuations)

### 4. Removing HTML tags

In [10]:
def remove_html(text):
    text = re.sub(r"<.*?>", "", text)
    return text

df["review"]=df["review"].apply(remove_html)

In [11]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


### 5. Removing Stopwords

In [12]:
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Utpala\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Utpala\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Utpala\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [13]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")
    for token in tokens:
        if token in stop_words:
            text = text.replace(token,"")
    return text

df["review"]=df["review"].apply(remove_stopwords)

### 6. Stemming

In [14]:
from nltk.stem import PorterStemmer

def stemming(text):
    stemmed_words=[]
    ps=PorterStemmer()
    tokens= word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)
    return " ".join(stemmed_words)

df["review"]=df["review"].apply(stemming)

In [15]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


### 7. Encoding

In [18]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df["sentiment"] = le.fit_transform(df["sentiment"])
y= df["sentiment"]

In [19]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

### 8. Vectorization

In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf= TfidfVectorizer(max_features = 5000)
X = tf.fit_transform(df["review"])

X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4057169 stored elements and shape (49582, 5000)>

## Dataset & Data Loaders

In [21]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train.shape

(39665, 5000)

In [23]:
import torch
from torch.utils.data import TensorDataset, DataLoader

X_train=X_train.toarray()
X_test=X_test.toarray()

train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)
test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

train_loader = DataLoader(train_set, batch_size = 64, shuffle=True)
test_loader = DataLoader(test_set, batch_size = 64, shuffle=True)
    

## Building RNN

In [24]:
import torch.nn as nn
import torch.optim as optim

class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()
        self.hidden_size=hidden_size
        self.num_layers= num_layers

        # RNN Layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first= True)

        # Fully Connected Layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        out,_ = self.rnn(x, h0)
        out = self.fc(out[:,-1,:])
        return out

In [27]:
input_size=X_train.shape[1]
model=RNN(input_size)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

## Training RNN

In [28]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction
        
        outputs = model(Xb) # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => probability

        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backprop
        optimizer.step() # weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.16114334762096405
epoch = 2/10 and loss = 0.2615923583507538
epoch = 3/10 and loss = 0.2644036114215851
epoch = 4/10 and loss = 0.35322096943855286
epoch = 5/10 and loss = 0.31135931611061096
epoch = 6/10 and loss = 0.1270657330751419
epoch = 7/10 and loss = 0.10109508037567139
epoch = 8/10 and loss = 0.4052863121032715
epoch = 9/10 and loss = 0.17088599503040314
epoch = 10/10 and loss = 0.261437326669693


## Evaluate

In [29]:
model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    
    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 85.56014923868105
